# Explore condition-specific entity similarity

Interactive POC for SimBench proposal §3.2. Choose a discovery run, condition, retrieved entity type, query entity, and signal set; the notebook will show similarity scores, component scores, and raw records.

Run the setup and interactive cells in order. Text embeddings are reused from a persistent SQLite cache under `/storage/lcostes/simbench-cache`; precompute the RelAmazon text once with `python scripts/precompute_text_embeddings.py` for faster first use. The button reports progress and prints any exception in its output area.


In [1]:
from pathlib import Path
import os
import json
import importlib
import sys
import traceback

import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display, Markdown

REPO = Path.cwd().resolve()
if not (REPO / "src").is_dir():
    REPO = Path("/storage/lcostes/Similarity_benchmark")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.datasets.local_relbench_dataset import LocalRelBenchDataset
from src.schema.schema_graph import SchemaGraph
from src.signals.candidates import candidate_from_dict
import src.similarity.signal_measures as signal_measures
signal_measures = importlib.reload(signal_measures)
DEFAULT_TEXT_EMBEDDING_CACHE = signal_measures.DEFAULT_TEXT_EMBEDDING_CACHE
SignalMeasureExecutor = signal_measures.SignalMeasureExecutor

DATASET_ROOT = REPO / "data/dev/rel-amazon"
RESULTS_ROOT = REPO / "results/rel-amazon/discovery"
CONDITIONS_ROOT = REPO / "conditions/rel-amazon"

MODEL_CACHE_ROOT = Path(os.environ.get("SIMBENCH_MODEL_CACHE_ROOT", "/storage/lcostes/model-cache/huggingface/hub"))
TEXT_EMBEDDING_CACHE = Path(os.environ.get("SIMBENCH_TEXT_EMBEDDING_CACHE", str(DEFAULT_TEXT_EMBEDDING_CACHE)))
dataset = LocalRelBenchDataset(DATASET_ROOT)
graph = SchemaGraph.from_manifest(dataset.schema)
executor = SignalMeasureExecutor(dataset, graph, model_cache_root=MODEL_CACHE_ROOT, text_embedding_cache=TEXT_EMBEDDING_CACHE)

DEFAULT_MEASURE_VARIANT = "auto"  # Change this to set the initial measure for every selected signal.
MEASURE_LABELS = {
    "auto": "",
    "jensen-shannon": "Jensen–Shannon",
    "hellinger": "Hellinger",
    "wl-subtree-kernel": "WL subtree kernel",
    "diffusion-distance": "One-step diffusion distance",
}
_MEASURE_OPTIONS_CACHE = {}
_AUTO_MEASURE_LABEL_CACHE = {}
def measure_options_for(candidate):
    if candidate.id not in _MEASURE_OPTIONS_CACHE:
        _MEASURE_OPTIONS_CACHE[candidate.id] = executor.available_measure_variants(candidate)
    if candidate.id not in _AUTO_MEASURE_LABEL_CACHE:
        _AUTO_MEASURE_LABEL_CACHE[candidate.id] = executor.automatic_measure_name(candidate)
    return [
        (f"{_AUTO_MEASURE_LABEL_CACHE[candidate.id]} (auto)" if key == "auto" else MEASURE_LABELS[key], key)
        for key in _MEASURE_OPTIONS_CACHE[candidate.id]
    ]
display(Markdown(f"Persistent text embedding cache: `{TEXT_EMBEDDING_CACHE}` ({'ready' if TEXT_EMBEDDING_CACHE.exists() else 'not precomputed yet'})"))

def load_jsonl(path):
    with Path(path).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def run_files():
    return sorted((RESULTS_ROOT / "catalog-views").glob("*.jsonl"))

def get_record(filename, condition_id):
    path = RESULTS_ROOT / "catalog-views" / filename
    return next((row for row in load_jsonl(path) if row.get("condition_id") == condition_id), None)

def ids_for(table):
    primary_key = dataset.get_primary_key(table)
    if primary_key is None:
        return tuple(range(dataset.get_table(table).num_rows)), "row position"
    return tuple(dataset.get_table(table)[primary_key].to_pylist()), "primary key"

def entity_row(table, entity_id):
    primary_key = dataset.get_primary_key(table)
    if primary_key is None:
        return dataset.get_table(table).slice(int(entity_id), 1).to_pylist()[0]
    ids = dataset.get_table(table)[primary_key].to_pylist()
    resolved_id = next(value for value in ids if str(value) == str(entity_id))
    return dataset.get_entity(table, resolved_id).to_pylist()[0]

def discovery_score_value(stored):
    value = stored.get("utility_score", stored.get("score"))
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None

def discovery_score_label(stored, score_type):
    utility = stored.get("utility_score")
    if utility is not None:
        scale = "10" if score_type.endswith("_0_10") else "100" if score_type.endswith("_0_100") else "?"
        return f"discovery score {utility}/{scale}"
    score = stored.get("score")
    if score is not None:
        return f"model score {float(score):.3g}"
    return "score unavailable"


Persistent text embedding cache: `/storage/lcostes/simbench-cache/text-embeddings.sqlite3` (ready)

## Choose a condition, discovery run, entity type, signals, and measures

Only signals whose paths start from the selected entity table are offered. Each selected signal gets its own measure dropdown, containing only compatible choices. The automatic option names the function it will use for that signal (for example, `Percentile Wasserstein similarity (auto)`). Change `DEFAULT_MEASURE_VARIANT` near the top of the setup cell to change the initial choice for every signal. Numeric and temporal distribution variants appear only when the path yields multiple values; graph variants are limited to relationship signals. Diffusion can return missing scores when there are no shared endpoints.


In [2]:
run_picker = widgets.Dropdown(description="Run:", layout=widgets.Layout(width="95%"))
condition_picker = widgets.Dropdown(description="Condition:", layout=widgets.Layout(width="95%"))
entity_picker = widgets.Dropdown(description="Entity:")
query_picker = widgets.Dropdown(description="Query ID:", layout=widgets.Layout(width="60%"))
query_preview = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
signal_picker = widgets.SelectMultiple(description="Signals:", rows=10, layout=widgets.Layout(width="95%"))
measure_controls = widgets.VBox()
measure_pickers = {}
weight_mode = widgets.ToggleButtons(
    options=[
        ("Equal weights (recommended)", "equal"),
        ("Relative discovery scores", "discovery"),
        ("Custom sliders", "custom"),
    ],
    value="equal",
    description="Weight preset:",
    style={"description_width": "initial"},
)
weight_help = widgets.HTML()
weight_controls = widgets.VBox()
top_k_picker = widgets.IntSlider(value=5, min=1, max=20, step=1, description="Show top k:")
run_button = widgets.Button(description="Score nearest entities", button_style="primary", icon="search")
status = widgets.HTML()
result_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
_busy = False


def refresh_controls(*_):
    global _busy
    if _busy:
        return
    _busy = True
    try:
        files = run_files()
        old_run = run_picker.value
        run_picker.options = [(path.name.replace("catalog-rank-", "Catalog rank — ").replace("-path-aware", " [path-aware]"), path.name) for path in files]
        run_picker.value = old_run if old_run in [path.name for path in files] else (files[0].name if files else None)

        available_ids = {row.get("condition_id") for row in load_jsonl(RESULTS_ROOT / "catalog-views" / run_picker.value)} if run_picker.value else set()
        condition_rows = [row for row in load_jsonl(CONDITIONS_ROOT / "conditions.jsonl") if row["id"] in available_ids]
        old_condition = condition_picker.value
        condition_picker.options = [(f"{row['id']} — {row['condition']}", row['id']) for row in condition_rows]
        condition_ids = [row["id"] for row in condition_rows]
        condition_picker.value = old_condition if old_condition in condition_ids else (condition_ids[0] if condition_ids else None)

        record = get_record(run_picker.value, condition_picker.value) if run_picker.value and condition_picker.value else None
        candidates = [candidate_from_dict(row) for row in record.get("signals", []) if isinstance(row, dict)] if record else []
        entity_tables = sorted({candidate.source_table for candidate in candidates})
        old_entity = entity_picker.value
        entity_picker.options = entity_tables
        entity_picker.value = old_entity if old_entity in entity_tables else (entity_tables[0] if entity_tables else None)

        if entity_picker.value:
            entity_ids, id_kind = ids_for(entity_picker.value)
            query_picker.options = [str(value) for value in entity_ids]
            if str(query_picker.value) not in query_picker.options:
                query_picker.value = str(entity_ids[0]) if entity_ids else None
        else:
            entity_ids, id_kind = (), ""
            query_picker.options = []
            query_picker.value = None

        compatible = [(candidate, stored) for candidate, stored in zip(candidates, record.get("signals", [])) if candidate.source_table == entity_picker.value] if record and entity_picker.value else []
        previous = set(signal_picker.value)
        signal_picker.options = [
            (f"#{stored.get('rank', '?')} | {discovery_score_label(stored, record.get('score_type', ''))} | {candidate.id} | {candidate.explain_for_human(graph)}", candidate.id)
            for candidate, stored in compatible
        ]
        allowed = {candidate.id for candidate, _ in compatible}
        retained = tuple(value for value in previous if value in allowed)
        signal_picker.value = retained or tuple(candidate.id for candidate, _ in compatible[:3])
        status.value = f"{len(compatible)} compatible signals rooted at <code>{entity_picker.value}</code>; {len(entity_ids):,} query entities available, IDs are <b>{id_kind}</b>."
        show_query_entity()
        rebuild_weight_controls()
    except Exception as error:
        status.value = f"<span style='color:#b91c1c'><b>Control setup error:</b> {error}</span>"
    finally:
        _busy = False


def show_query_entity(*_):
    with query_preview:
        clear_output(wait=True)
        if not entity_picker.value or query_picker.value in (None, ""):
            display(Markdown("Select an entity table and query ID to preview its attributes."))
            return
        row = entity_row(entity_picker.value, query_picker.value)
        display(Markdown(f"**Query entity:** `{entity_picker.value}` / `{query_picker.value}`"))
        display(pd.DataFrame([{"attribute": key, "value": value} for key, value in row.items()]))


weight_sliders = {}
_updating_weights = False

def selected_signal_records():
    record = get_record(run_picker.value, condition_picker.value)
    if record is None:
        return {}, ""
    return {
        row["id"]: row for row in record.get("signals", [])
        if isinstance(row, dict) and row.get("source_table") == entity_picker.value
    }, str(record.get("score_type", ""))

def update_weight_help():
    if weight_mode.value == "equal":
        weight_help.value = "All selected signals have weight 1. This is the baseline to compare against."
    elif weight_mode.value == "discovery":
        weight_help.value = "Weights are integers from 1 to 10, relative to the highest selected discovery score. Use this as an experiment: model discovery scores are not calibrated relevance probabilities."
    else:
        weight_help.value = "Adjust each selected signal from 0 to 10. A weight of 0 removes its contribution from the combined similarity."

def apply_weight_mode(*_):
    global _updating_weights
    _updating_weights = True
    try:
        records, _ = selected_signal_records()
        if weight_mode.value == "equal":
            values = {signal_id: 1 for signal_id in weight_sliders}
        elif weight_mode.value == "discovery":
            raw_scores = {signal_id: discovery_score_value(records.get(signal_id, {})) for signal_id in weight_sliders}
            positive_scores = [score for score in raw_scores.values() if score is not None and score > 0]
            maximum = max(positive_scores, default=0.0)
            values = {
                signal_id: max(1, min(10, round(10 * score / maximum))) if score is not None and maximum else 1
                for signal_id, score in raw_scores.items()
            }
        else:
            values = {}
        for signal_id, value in values.items():
            weight_sliders[signal_id].value = value
        update_weight_help()
    finally:
        _updating_weights = False

def slider_changed(_change):
    if not _updating_weights and weight_mode.value != "custom":
        weight_mode.value = "custom"

def rebuild_weight_controls(*_):
    global weight_sliders, measure_pickers
    previous = {signal_id: slider.value for signal_id, slider in weight_sliders.items()}
    previous_measures = {signal_id: picker.value for signal_id, picker in measure_pickers.items()}
    records, score_type = selected_signal_records()
    selected_ids = list(signal_picker.value)
    weight_sliders = {}
    measure_pickers = {}
    controls = []
    measure_rows = []
    for signal_id in selected_ids:
        stored = records.get(signal_id)
        if stored is None:
            continue
        rank = stored.get("rank", "?")
        candidate = candidate_from_dict(stored)
        choices = measure_options_for(candidate)
        allowed_measures = [value for _, value in choices]
        selected_measure = previous_measures.get(signal_id, DEFAULT_MEASURE_VARIANT)
        if selected_measure not in allowed_measures:
            selected_measure = "auto"
        measure_picker = widgets.Dropdown(
            options=choices, value=selected_measure, description="Measure:",
            style={"description_width": "initial"}, layout=widgets.Layout(width="95%"),
        )
        measure_pickers[signal_id] = measure_picker
        measure_rows.append(widgets.VBox([
            widgets.HTML(f"<b>#{rank}</b> · <code>{signal_id}</code>"), measure_picker,
        ]))
        slider = widgets.IntSlider(
            value=previous.get(signal_id, 1), min=0, max=10, step=1,
            description="weight:", style={"description_width": "initial"},
            layout=widgets.Layout(width="55%"),
        )
        slider.observe(slider_changed, names="value")
        weight_sliders[signal_id] = slider
        controls.append(widgets.VBox([
            widgets.HTML(f"<b>#{rank}</b> · {discovery_score_label(stored, score_type)}<br><code>{signal_id}</code>"),
            slider,
        ]))
    measure_controls.children = tuple(measure_rows) or (widgets.HTML("Select one or more signals to choose their measures."),)
    weight_controls.children = tuple(controls) or (widgets.HTML("Select one or more signals to set their weights."),)
    apply_weight_mode()

def current_weights(selected_ids):
    return {signal_id: weight_sliders[signal_id].value for signal_id in selected_ids if signal_id in weight_sliders}


def score_nearest(_button):
    run_button.disabled = True
    status.value = "Preparing selected signals and scoring candidates…"
    with result_output:
        clear_output(wait=True)
        try:
            filename = run_picker.value
            condition_id, entity_table = condition_picker.value, entity_picker.value
            record = get_record(filename, condition_id)
            if record is None:
                raise ValueError("This run has no result for the selected condition.")
            stored_by_id = {
                row["id"]: row for row in record.get("signals", [])
                if isinstance(row, dict) and row.get("source_table") == entity_table
            }
            selected_ids = list(signal_picker.value)
            if not selected_ids:
                raise ValueError("Select at least one compatible signal.")
            missing = [signal_id for signal_id in selected_ids if signal_id not in stored_by_id]
            if missing:
                raise ValueError(f"Signals are not rooted at {entity_table}: {missing}")
            candidates = [candidate_from_dict(stored_by_id[signal_id]) for signal_id in selected_ids]
            weights = current_weights(selected_ids)
            materialized = [executor.materialize(candidate, measure_variant=measure_pickers[candidate.id].value) for candidate in candidates]
            ranked = executor.score_query(query_picker.value, materialized, weights=weights)
            query_id = ranked[0].query_entity_id if ranked else query_picker.value

            condition_row = next(row for row in load_jsonl(CONDITIONS_ROOT / "conditions.jsonl") if row["id"] == condition_id)
            display(Markdown(f"## {condition_id} — {entity_table} query `{query_id}`\n\n**Condition:** {condition_row['condition']}"))
            evidence_rows = []
            for signal in materialized:
                evidence_rows.append({
                    "rank": stored_by_id[signal.candidate.id].get("rank"),
                    "signal": signal.candidate.id,
                    "meaning": signal.candidate.explain_for_human(graph),
                    "measure": signal.measure,
                    "weight": weights.get(signal.candidate.id, 1.0),
                    "query values": ", ".join(map(str, signal.values_for(query_id)[:6])),
                })
            display(Markdown("### Selected evidence"))
            display(pd.DataFrame(evidence_rows).sort_values("rank"))
            display(Markdown("### Query record"))
            display(pd.DataFrame([entity_row(entity_table, query_id)]))

            result_rows, component_rows = [], []
            for rank, pair in enumerate(ranked[:top_k_picker.value], start=1):
                available = [component for component in pair.components if component.get("score") is not None]
                result_rows.append({
                    "rank": rank,
                    "candidate entity": pair.candidate_entity_id,
                    "combined similarity": pair.similarity_score,
                    "available signals": f"{len(available)}/{len(pair.components)}",
                })
                component_rows.extend({
                    "rank": rank,
                    "candidate entity": pair.candidate_entity_id,
                    "signal": component["signal_id"],
                    "component similarity": component["score"],
                    "evidence confidence": component.get("confidence", 1.0),
                    "effective weight": component.get("effective_weight", component.get("weight", 1.0)),
                } for component in available)
            display(Markdown("### Nearest entities"))
            display(pd.DataFrame(result_rows).round(3))
            display(Markdown("### Available component scores"))
            display(pd.DataFrame(component_rows).round(3) if component_rows else pd.DataFrame({"note": ["No selected signal had values for the returned entities."]}))
            display(Markdown("### Candidate records"))
            display(pd.DataFrame([
                {"candidate entity": pair.candidate_entity_id, "similarity": pair.similarity_score, **{key: str(value)[:240] for key, value in entity_row(entity_table, pair.candidate_entity_id).items()}}
                for pair in ranked[:top_k_picker.value]
            ]).round(3))
            status.value = f"Scored {min(len(ranked), top_k_picker.value)} candidates across {len(materialized)} signal(s)."
        except Exception as error:
            traceback.print_exc()
            status.value = f"<span style='color:#b91c1c'><b>Scoring error:</b> {error}</span>"
            display(Markdown(f"**Scoring error:** `{error}`"))
        finally:
            run_button.disabled = False


for control in (run_picker, condition_picker, entity_picker):
    control.observe(refresh_controls, names="value")
query_picker.observe(show_query_entity, names="value")
signal_picker.observe(rebuild_weight_controls, names="value")
weight_mode.observe(apply_weight_mode, names="value")
run_button.on_click(score_nearest)
refresh_controls()
display(widgets.VBox([
    run_picker, condition_picker, entity_picker, query_picker,
    query_preview, status, signal_picker, measure_controls, weight_mode, weight_help, weight_controls,
    top_k_picker, run_button, result_output,
]))
